# 전처리 결과 읽기와 기본 EDA

이 노트북은 **전처리를 다시 실행하지 않고** 저장된 Parquet 결과를 읽습니다. 위에서부터 차례로 실행하세요.

- 저장 위치: `MBTI_project/full_preprocessed/`
- 빠른 확인용 결과: `MBTI_project/full_preprocessed_preview_v2/`
- `classifier_strict.parquet`: MBTI 관련 표현까지 제거한 분류 모델용 댓글
- `classifier_base.parquet`: 일반 정제까지만 적용한 댓글
- `rag_train.parquet`: 학습 사용자에 한정한 RAG 후보
- `users.parquet`: 익명화된 사용자, MBTI, 사용자 단위 분할
- `audit.parquet`: 제외된 댓글과 사유를 포함한 전체 기록
- `report.json`: 전처리 집계

**단위 주의:** 원본 1행에는 여러 댓글이 이어져 있어, 원본 행 수와 분리 후 댓글 수가 다릅니다. 분류용 파일은 크므로 기본 EDA에서는 텍스트 전체를 메모리에 올리지 않습니다.

In [ ]:
from pathlib import Path
from collections import Counter
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# 미리보기 데이터를 보고 싶다면 True로 변경
USE_PREVIEW = False

project_dir = Path.cwd()
if not (project_dir / "full_preprocessed").is_dir():
    project_dir = Path("/Users/jinsehyeon/pyproject/MBTI_project")
data_dir = project_dir / ("full_preprocessed_preview_v2" if USE_PREVIEW else "full_preprocessed")
if not (data_dir / "report.json").exists():
    raise FileNotFoundError(f"전처리 결과가 없습니다: {data_dir}")

pd.set_option("display.max_columns", 30)
print("프로젝트:", project_dir)
print("읽는 폴더:", data_dir)

## 1. 파일별 행 수와 저장 크기

Parquet 메타데이터만 읽으므로 대용량 파일도 빠르게 확인할 수 있습니다.

In [ ]:
file_names = [
    "users.parquet", "audit.parquet", "classifier_base.parquet",
    "classifier_strict.parquet", "rag_train.parquet",
]
file_rows = []
for name in file_names:
    path = data_dir / name
    if path.exists():
        pf = pq.ParquetFile(path)
        file_rows.append({
            "파일": name,
            "행 수": pf.metadata.num_rows,
            "크기 (MiB)": round(path.stat().st_size / 1024**2, 1),
            "열 수": len(pf.schema.names),
        })
file_summary = pd.DataFrame(file_rows)
display(file_summary.style.format({"행 수": "{:,}", "크기 (MiB)": "{:,.1f}"}))

## 2. 전처리 보고서와 보존율

원본 데이터 행 수, 사용자 수, 댓글 제외 사유를 확인합니다.

In [ ]:
report = json.loads((data_dir / "report.json").read_text(encoding="utf-8"))
overview = pd.DataFrame([
    ("원본 행 수", report.get("source_rows")),
    ("원본 train/test에 중복 등장한 작성자", report.get("source_train_test_author_overlap")),
    ("서로 다른 MBTI가 기록된 작성자", report.get("conflicting_authors")),
    ("최종 사용자 수", report.get("authors_kept")),
], columns=["항목", "값"])
display(overview.style.format({"값": "{:,}"}))

counts = report.get("counts", {})
count_table = pd.DataFrame(
    [{"항목": key, "댓글 수": value} for key, value in counts.items()]
)
display(count_table.style.format({"댓글 수": "{:,}"}))

raw_comments = counts.get("raw_comments", 0)
if raw_comments:
    for name in ("classifier_base", "classifier_strict", "rag_train"):
        print(f"{name}: {counts.get(name, 0):,} / {raw_comments:,} = {counts.get(name, 0)/raw_comments:.1%}")

## 3. 사용자별 유형과 데이터 분할

`users.parquet`은 작아서 한 번에 읽습니다. 분할은 댓글이 아니라 **사용자 단위**입니다.

In [ ]:
users = pd.read_parquet(data_dir / "users.parquet")
display(users.head())
print(f"사용자 수: {users['user_id'].nunique():,}")
print(f"중복 user_id: {users['user_id'].duplicated().sum():,}")
print(f"MBTI 유형 수: {users['mbti'].nunique():,}")
user_counts = pd.crosstab(users["mbti"], users["split"])
user_counts["전체"] = user_counts.sum(axis=1)
display(user_counts.sort_index().style.format("{:,}"))
if "train" in users["split"].values and "test" in users["split"].values:
    train_ids = set(users.loc[users["split"] == "train", "user_id"])
    test_ids = set(users.loc[users["split"] == "test", "user_id"])
    print("train/test 사용자 겹침:", len(train_ids & test_ids))

## 4. 분류용 댓글의 유형·분할별 행 수

텍스트 열을 읽지 않고 필요한 열만 배치로 읽습니다. 전체 결과에서는 잠시 걸릴 수 있습니다.

In [ ]:
def count_by_type_split(parquet_path, batch_size=250_000):
    counter = Counter()
    total_words = 0
    total_rows = 0
    min_words = None
    max_words = None
    for batch in pq.ParquetFile(parquet_path).iter_batches(
        batch_size=batch_size, columns=["mbti", "split", "word_count"]
    ):
        frame = batch.to_pandas()
        counter.update(frame.groupby(["mbti", "split"], dropna=False).size().to_dict())
        words = frame["word_count"].dropna()
        total_words += words.sum()
        total_rows += len(frame)
        if not words.empty:
            min_words = min(words.min(), min_words) if min_words is not None else words.min()
            max_words = max(words.max(), max_words) if max_words is not None else words.max()
    table = pd.Series(counter, name="댓글 수")
    table.index = pd.MultiIndex.from_tuples(table.index, names=["mbti", "split"])
    table = table.unstack(fill_value=0).sort_index()
    table["전체"] = table.sum(axis=1)
    stats = {"댓글 수": total_rows, "평균 단어 수": total_words / total_rows if total_rows else np.nan,
             "최소 단어 수": min_words, "최대 단어 수": max_words}
    return table, stats

strict_counts, strict_stats = count_by_type_split(data_dir / "classifier_strict.parquet")
display(strict_counts.style.format("{:,}"))
display(pd.DataFrame([strict_stats]).style.format({
    "댓글 수": "{:,.0f}", "평균 단어 수": "{:,.1f}",
    "최소 단어 수": "{:,.0f}", "최대 단어 수": "{:,.0f}",
}))
print("유형별 최대/최소 댓글 수 비율:",
      round(strict_counts["전체"].max() / strict_counts["전체"].min(), 2))

## 5. 사용자당 댓글 수

유형별 댓글 수가 비슷해도 소수 사용자의 댓글이 지나치게 많을 수 있습니다. 작성자별 댓글 수 분포를 확인합니다.

In [ ]:
def count_by_user(parquet_path, batch_size=250_000):
    counter = Counter()
    for batch in pq.ParquetFile(parquet_path).iter_batches(
        batch_size=batch_size, columns=["user_id"]
    ):
        counter.update(batch.column("user_id").to_pylist())
    return pd.Series(counter, name="댓글 수")

comments_per_user = count_by_user(data_dir / "classifier_strict.parquet")
display(comments_per_user.describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_frame().style.format("{:,.1f}"))
print("댓글이 남은 사용자:", f"{len(comments_per_user):,}")
print("상위 1% 사용자의 댓글 비중:",
      f"{comments_per_user.nlargest(max(1, round(len(comments_per_user) * .01))).sum() / comments_per_user.sum():.1%}")
print("5개 이하 댓글 사용자:", f"{(comments_per_user <= 5).sum():,}")

## 6. 제외 사유

`audit.parquet`에서 사유 열만 읽습니다. 이 값은 *분리된 댓글* 기준입니다.

In [ ]:
reason_counts = Counter()
for batch in pq.ParquetFile(data_dir / "audit.parquet").iter_batches(
    batch_size=250_000, columns=["reason"]
):
    reason_counts.update(batch.column("reason").to_pylist())
reason_table = pd.DataFrame(
    [{"사유": key or "(통과)", "댓글 수": value} for key, value in reason_counts.items()]
).sort_values("댓글 수", ascending=False)
reason_table["비율"] = reason_table["댓글 수"] / reason_table["댓글 수"].sum()
display(reason_table.style.format({"댓글 수": "{:,}", "비율": "{:.1%}"}))

## 7. 실제 데이터 읽기

아래 첫 코드는 텍스트 5개만 읽습니다. 학습할 때는 필요한 열과 분할을 선택해 읽으세요. 전체 `classifier_strict.parquet`을 한 번에 읽으면 RAM을 많이 사용할 수 있습니다.

In [ ]:
strict_file = data_dir / "classifier_strict.parquet"
sample = next(pq.ParquetFile(strict_file).iter_batches(
    batch_size=5, columns=["user_id", "mbti", "split", "text", "word_count"]
)).to_pandas()
display(sample)

In [ ]:
# 학습 데이터 전체를 메모리에 올릴 수 있을 때만 실행하세요.
# train = pd.read_parquet(
#     strict_file,
#     columns=["user_id", "mbti", "split", "text"],
#     filters=[("split", "=", "train")],
# )
# test = pd.read_parquet(
#     strict_file,
#     columns=["user_id", "mbti", "split", "text"],
#     filters=[("split", "=", "test")],
# )
# print(len(train), len(test))

# RAM이 부족하면 아래처럼 배치로 순회하세요.
# for batch in pq.ParquetFile(strict_file).iter_batches(
#     batch_size=50_000, columns=["user_id", "mbti", "split", "text"]
# ):
#     chunk = batch.to_pandas()
#     train_chunk = chunk[chunk["split"] == "train"]
#     # 필요한 특징 추출 또는 파일 저장
#     del chunk, train_chunk